# Laboratorio 4 · Asistente de compras por voz

El asistente recibe consultas en español, recupera fichas de ESCI, administra una tienda simulada y responde con una voz sintética clonada a partir de una referencia. La ejecución de modelos y los resultados de entrega se obtienen en un kernel de Google Colab. El notebook incluye los módulos de Python y descarga los datos y pesos públicos al ejecutarse en Colab. La carpeta de VS Code conserva los scripts editables y las verificaciones locales.

| Bloque | Evidencia |
|---|---|
| Datos e índice | Catálogo de al menos 5.000 fichas y cobertura íntegra de 200 consultas de test. |
| Recuperación | BM25, E5 sin BM25 y método híbrido, sobre los mismos candidatos. |
| Agente y voz | Estado de conversación, transacciones verificables, ASR y clonación de voz. |
| Evaluación | Veinte escenarios, seis con audio y control por texto, latencias y evidencias por turno. |


In [ ]:
#@title Preparar el notebook y sus módulos
from pathlib import Path
from base64 import b64decode
from hashlib import sha256
from datetime import datetime, timezone
from importlib import metadata
from uuid import uuid4
import io
import json
import os
import platform
import shutil
import subprocess
import sys
import zipfile

REQUERIDOS = {"asistente.py", "esci.py", "evaluacion.py", "modelos.py", "requirements.txt",
              "tienda.py", "verificar_local.py"}

SHA_FUENTES = "b7daa2d58334642ab86b8d97acda71337c9c9f60bd8c0fc15187175fa3d31d21"
FUENTES = (
    "..."
)

try:
    import google.colab
    EN_COLAB = sys.platform == "linux" and (Path("/content").is_dir() or "COLAB_RELEASE_TAG" in os.environ)
except ImportError:
    EN_COLAB = False

# Los archivos editables se usan al abrir la carpeta del proyecto desde VS Code.
# El paquete de fuentes permite ejecutar este mismo notebook por sí solo en Colab.
ROOT = Path.cwd()
if EN_COLAB or not all((ROOT / nombre).is_file() for nombre in REQUERIDOS):
    ROOT = (Path("/content") if EN_COLAB else ROOT) / "Laboratorio4_ESCI"
    comprimido = b64decode(FUENTES)
    if sha256(comprimido).hexdigest() != SHA_FUENTES:
        raise ValueError("El paquete de fuentes del notebook no supera la verificación SHA-256.")
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(comprimido)) as fuente:
        if set(fuente.namelist()) != REQUERIDOS or len(fuente.namelist()) != len(REQUERIDOS):
            raise ValueError("El archivo integrado de fuentes no tiene la estructura prevista.")
        for nombre in sorted(REQUERIDOS):
            (ROOT / nombre).write_bytes(fuente.read(nombre))

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
ID_EJECUCION = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
RESULTADOS = Path("resultados") / ID_EJECUCION
if EN_COLAB:
    RESULTADOS.mkdir(parents=True)
print("Carpeta:", ROOT, "| Kernel de Colab:", EN_COLAB)


## 1 · Dependencias y verificación de desarrollo

La prueba local usa tres fichas ficticias y ejecuta las herramientas por texto. Permite comprobar, entre otros casos, que una segunda confirmación devuelve el mismo pedido y que un fallo durante una compra con dos productos revierte ambas escrituras.


In [ ]:
if EN_COLAB:
    print("Python del entorno:", platform.python_version())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if shutil.which("sox") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "sox"], check=True)

from verificar_local import comprobaciones
control = comprobaciones()
print(f"Control de desarrollo por texto: {control['escenarios_correctos']}/{control['escenarios']}")
print("Comprobaciones:", ", ".join(control["comprobaciones"]))


## 2 · Preparación de ESCI

Se conserva la separación oficial por ID de consulta y el filtro del mercado español para la versión reducida. Se toman 60 consultas de desarrollo y 200 de evaluación, con muestreo estratificado por número de candidatos. El catálogo incluye la unión de los candidatos anotados. Si se carga Laboratorio4_ESCI.zip en /content, el notebook utiliza el subconjunto de la entrega y verifica sus huellas. La preparación también admite descargar ambos Parquet originales y verificar su SHA-256. El muestreo fijado produjo 6.915 fichas, 5.470 pares anotados de test y cobertura completa de las 200 consultas. El notebook informa el resultado de su propia preparación.

El agente recibe el catálogo de fichas. La evaluación lee las etiquetas de relevancia desde tablas independientes.


In [ ]:
if EN_COLAB:
    import numpy as np
    import pandas as pd
    import torch
    from IPython.display import Audio, display
    from esci import preparar_datos, SEED

    np.random.seed(SEED)
    torch.manual_seed(SEED)
    datos_dir = Path("datos")
    paquete = Path("/content/Laboratorio4_ESCI.zip")
    preparados = datos_dir / "esci_es_200_60_42"
    if paquete.is_file() and not (preparados / "resumen.json").is_file():
        preparados.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(paquete) as zip_datos:
            prefijo = "datos/esci_es_200_60_42"
            if f"{prefijo}/resumen.json" not in zip_datos.namelist():
                prefijo = f"Laboratorio4_ESCI/{prefijo}"
            for nombre in ("catalogo.parquet", "desarrollo.parquet", "test.parquet", "LICENSE", "NOTICE", "resumen.json"):
                (preparados / nombre).write_bytes(zip_datos.read(f"{prefijo}/{nombre}"))
    datos = preparar_datos(datos_dir, n_test=200, n_dev=60, semilla=SEED)
    display(pd.Series(datos.resumen, name="valor").to_frame())
    assert len(datos.catalogo) >= 5000
    assert datos.test.query_id.nunique() >= 200
    assert set(datos.test.product_id).issubset(datos.catalogo.product_id)
    assert not set(datos.test.query_id).intersection(datos.desarrollo.query_id)
    (RESULTADOS / "datos.json").write_text(json.dumps(datos.resumen, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    print("Use un kernel de Colab para la descarga y la evaluación de entrega.")


## 3 · Recuperación y ablación

BM25 es la base léxica. El índice E5 obtiene representaciones semánticas multilingües. El método completo fusiona las dos listas mediante suma recíproca de posiciones. La versión E5 sin BM25 retira la rama léxica y conserva el resto del protocolo. Los parámetros se fijan antes de medir test.

La evaluación puntúa todos los candidatos anotados de cada consulta. nDCG@10 usa la escala E 1, S 0,1, C 0,01 e I 0. En empates se ordena por identificador.


In [ ]:
if EN_COLAB:
    from time import perf_counter
    import matplotlib.pyplot as plt
    from esci import Indice, evaluar_recuperacion, METODOS
    from modelos import E5, dispositivo
    from evaluacion import analizar_ranking

    equipo = dispositivo()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        print("GPU asignada:", torch.cuda.get_device_name(0))
    else:
        print("Kernel en CPU. La síntesis de voz y los modelos serán más lentos.")
    encoder = E5(equipo)
    inicio_indice = perf_counter()
    indice = Indice(datos.catalogo, encoder, "datos/cache_indice")
    indice_s = perf_counter() - inicio_indice
    r_dev, _ = evaluar_recuperacion(indice, datos.desarrollo)
    r_test, top10 = evaluar_recuperacion(indice, datos.test)

    for split, resultados in [("desarrollo", r_dev), ("test", r_test)]:
        resultados["split"] = split
    pd.concat([r_dev, r_test]).to_csv(RESULTADOS / "ranking_por_consulta.csv", index=False)
    top10.to_csv(RESULTADOS / "top10_test.csv", index=False, encoding="utf-8-sig")
    medida = r_test.groupby("metodo")["ndcg10"].agg(["count", "mean", "median", "std"]).reindex(METODOS)
    display(medida.round(4))
    por_consulta = r_test.pivot(index="query_id", columns="metodo", values="ndcg10")
    delta = pd.DataFrame({
        "hibrido_menos_bm25": por_consulta[METODOS[2]] - por_consulta[METODOS[0]],
        "hibrido_menos_ablacion": por_consulta[METODOS[2]] - por_consulta[METODOS[1]],
    })
    delta.to_csv(RESULTADOS / "diferencias_pareadas.csv")
    comparacion = {
        "consultas": int(por_consulta.shape[0]),
        "ndcg10_por_metodo": medida.to_dict(orient="index"),
        "diferencias_pareadas": {
            columna: {"media": float(valores.mean()), "mejora": int(valores.gt(1e-9).sum()),
                      "empate": int(valores.abs().le(1e-9).sum()), "cae": int(valores.lt(-1e-9).sum())}
            for columna, valores in delta.items()
        },
    }
    (RESULTADOS / "resumen_recuperacion.json").write_text(
        json.dumps(comparacion, ensure_ascii=False, indent=2), encoding="utf-8",
    )
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    medida["mean"].plot.barh(ax=ax, color=["#516783", "#67a69b", "#227566"])
    ax.set(xlabel="nDCG@10 promedio", ylabel="", xlim=(0, 1))
    ax.grid(axis="x", alpha=0.2)
    fig.tight_layout()
    fig.savefig(RESULTADOS / "ndcg10.png", dpi=160, bbox_inches="tight")
    plt.show()
    aciertos_fallos = analizar_ranking(r_test, top10, datos.test, indice.fichas, RESULTADOS)
    display(aciertos_fallos)


## 4 · Agente, voz y estado de tienda

El modelo de lenguaje propone operaciones, y el controlador valida la confirmación, las cantidades, los importes y el inventario. SQLite registra el pedido y el descuento de existencias en una sola transacción.

VoiceDesign sintetiza por defecto una referencia original y crea las intervenciones de prueba con otra voz. La ruta WAV y el texto exacto permiten usar una grabación propia o autorizada. Después se libera ese modelo y se carga el clonador que toma la grabación de referencia. Faster‑Whisper transcribe las intervenciones en CPU para reducir el uso de VRAM.

Las fichas de los guiones de operación son coincidencias exactas de dos consultas de desarrollo. Durante el diálogo, el agente solo accede a fichas y datos de la tienda.


In [ ]:
#@title Compatibilidad de PyAV y precisión de voz
import av
import importlib
import modelos

ruta_modelo = ROOT / "modelos.py"
sha_original = "9e18407e4a09e6a12496650c6c322eac4049a33a4cad27c66d942737e1148092"
sha_corregido = "b6e54174aa31de038f703c8aa214de120b49fed3bc058304d6ffd8259a353e38"
sha_estable = "a3b000897adf8666cf6b56b63f1fd0294fc805ebccd07766d490440c102cc8d3"
sha_voz = "27662938be1a8aec646d6c2aefa0e894dd80f33b1945b5738707804cf81a0776"
contenido = ruta_modelo.read_bytes()
actual = sha256(contenido).hexdigest()
if actual == sha_original:
    helper = '''def compatibilidad_pyav():
    """Adapta el decodificador de Faster-Whisper a la API de PyAV 19."""
    import av
    from faster_whisper import audio

    if int(av.__version__.split(".", 1)[0]) < 19 or audio.av is not av:
        return

    class BackendAudio:
        def __getattr__(self, nombre):
            return getattr(av, nombre)

        def open(self, *args, **opciones):
            opciones.pop("metadata_errors", None)
            return av.open(*args, **opciones)

    audio.av = BackendAudio()


'''
    codigo = contenido.decode("utf-8").replace("class Whisper:\n", helper + "class Whisper:\n", 1)
    codigo = codigo.replace("        from faster_whisper import WhisperModel\n        t = perf_counter()", "        from faster_whisper import WhisperModel\n        compatibilidad_pyav()\n        t = perf_counter()", 1)
    nuevo = codigo.encode("utf-8")
    assert sha256(nuevo).hexdigest() == sha_corregido
    ruta_modelo.write_bytes(nuevo)
else:
    assert actual in (sha_corregido, sha_estable, sha_voz)

codigo = ruta_modelo.read_bytes()
if sha256(codigo).hexdigest() == sha_corregido:
    anterior = '''def _audio_model(modelo, device):
    from qwen_tts import Qwen3TTSModel
    return Qwen3TTSModel.from_pretrained(
        modelo, device_map=device, dtype=dtype_modelo(device), attn_implementation="sdpa"
    )'''
    estable = '''def _audio_model(modelo, device):
    import torch
    from qwen_tts import Qwen3TTSModel

    precision = dtype_modelo(device)
    if modelo == CLON_ID and precision == torch.float16:
        # La predicción de códigos del clonador puede desbordarse en FP16 en Turing.
        precision = torch.float32
    return Qwen3TTSModel.from_pretrained(
        modelo, device_map=device, dtype=precision, attn_implementation="sdpa"
    )'''
    corregido = codigo.decode("utf-8").replace(anterior, estable, 1).encode("utf-8")
    assert sha256(corregido).hexdigest() == sha_estable
    ruta_modelo.write_bytes(corregido)

if sha256(ruta_modelo.read_bytes()).hexdigest() == sha_estable:
    fuente = ruta_modelo.read_text(encoding="utf-8")
    previo = "        wavs, sr = self.modelo.generate_voice_clone(\n            text=normal, language=\"Spanish\", voice_clone_prompt=self.prompt, max_new_tokens=1100,\n            do_sample=False,\n        )"
    corregido = "        # El presupuesto crece con el texto y acota repeticiones sin token de fin.\n        max_tokens = max(96, 36 + 7 * len(normal.split()))\n        wavs, sr = self.modelo.generate_voice_clone(\n            text=normal, language=\"Spanish\", voice_clone_prompt=self.prompt,\n            max_new_tokens=max_tokens, do_sample=True, non_streaming_mode=True,\n        )"
    fuente = fuente.replace(previo, corregido, 1).encode("utf-8")
    assert sha256(fuente).hexdigest() == sha_voz
    ruta_modelo.write_bytes(fuente)
    parcial = RESULTADOS / "operacion"
    diagnostico = ROOT / "diagnosticos" / f"{ID_EJECUCION}_tts_previo"
    if parcial.is_dir() and not diagnostico.exists():
        diagnostico.parent.mkdir(exist_ok=True)
        parcial.rename(diagnostico)
assert sha256(ruta_modelo.read_bytes()).hexdigest() == sha_voz

importlib.reload(modelos).compatibilidad_pyav()
marca_audio = ROOT / "sesion_entradas_sinteticas.txt"
ORIGEN_ENTRADAS = None
if (RESULTADOS / "referencia").is_dir() and (RESULTADOS / "entradas").is_dir():
    marca_audio.write_text(ID_EJECUCION, encoding="utf-8")
elif marca_audio.is_file():
    anterior_id = marca_audio.read_text(encoding="utf-8").strip()
    origen = ROOT / "resultados" / anterior_id
    if (origen / "referencia").is_dir() and (origen / "entradas").is_dir():
        for directorio in ("referencia", "entradas"):
            shutil.copytree(origen / directorio, RESULTADOS / directorio, dirs_exist_ok=True)
        ORIGEN_ENTRADAS = anterior_id
        print("Entradas sintéticas reutilizadas de:", ORIGEN_ENTRADAS)
print(f"PyAV {av.__version__}. Clonación en float32 en T4 y síntesis configurada.")


In [ ]:
if EN_COLAB:
    from evaluacion import anclas_del_catalogo, definir_casos, entradas_sinteticas
    from modelos import (ASR_ID, AGENTE_ID, CLON_ID, DISENO_ID, E5_ID, PlanificadorQwen, VozClonada,
                          Whisper, generar_audios_sinteticos, dtype_modelo)

    a, b, c, familia, otra_familia = anclas_del_catalogo(indice, datos.desarrollo)
    anclas = (a, b, c)
    casos = definir_casos(a, b, c, familia, otra_familia)
    assert len(casos) == 20 and sum(c.voz for c in casos) == 6
    assert sum(c.voz and len(c.turnos) > 1 for c in casos) >= 3
    FUENTE_ENTRADA = "sintetica"
    ENTRADAS = RESULTADOS / "entradas"
    grabaciones = entradas_sinteticas(casos, ENTRADAS) if FUENTE_ENTRADA == "sintetica" else []
    RUTA_REFERENCIA_AUTORIZADA = ""  # @param {type:"string"}
    TEXTO_REFERENCIA_AUTORIZADA = ""  # @param {type:"string"}
    ruta_voz = RUTA_REFERENCIA_AUTORIZADA.strip()
    texto_voz = TEXTO_REFERENCIA_AUTORIZADA.strip()
    if bool(ruta_voz) != bool(texto_voz):
        raise ValueError("Indica la ruta WAV y el texto exacto de la referencia autorizada.")
    referencia_elegida = (ruta_voz, texto_voz) if ruta_voz else None
    paquete_audios = Path("/content/Audios_sinteticos_Lab4_sesion.zip")
    if paquete_audios.is_file():
        rutas_audio = [ruta for ruta, _ in grabaciones]
        if referencia_elegida is None:
            rutas_audio.extend(RESULTADOS / "referencia" / nombre for nombre in
                               ("referencia_asistente.wav", "referencia_asistente.json"))
        with zipfile.ZipFile(paquete_audios) as paquete_voz:
            datos_audio = {ruta: paquete_voz.read(ruta.relative_to(RESULTADOS).as_posix())
                           for ruta in rutas_audio}
        for ruta, contenido in datos_audio.items():
            ruta.parent.mkdir(parents=True, exist_ok=True)
            ruta.write_bytes(contenido)
        if grabaciones:
            ORIGEN_ENTRADAS = "20261122T225139Z_c8961921"
        print("Audios sintéticos de validación recuperados.", flush=True)
    referencia, metadatos_voz = generar_audios_sinteticos(
        RESULTADOS / "referencia", grabaciones, SEED, equipo, referencia_elegida,
    )
    whisper = Whisper()
    planificador = PlanificadorQwen(equipo)
    voz = VozClonada(referencia, metadatos_voz["texto"], equipo)
    print("Familias de productos de desarrollo:", familia, "y", otra_familia)
    print("Referencia:", metadatos_voz["origen"], "| duración:", round(metadatos_voz["duracion_s"], 1), "s")
    display(Audio(filename=str(referencia)))


## 5 · Demostración interactiva

Ejecuta **Preparar la conversación** para habilitar los controles. En **Configurar mi voz para las respuestas**, puedes grabar una referencia propia o continuar con la voz activa. Graba cada instrucción en **Conversación por micrófono** y pulsa **Finalizar conversación** al terminar.

La latencia se mide desde el fin de la grabación hasta que la respuesta se puede reproducir en el navegador. También se puede procesar un archivo con `turno_wav(ruta)`. En una prueba por WAV, puedes simular que se agota el stock después de cotizar dos unidades de la primera opción con `demo_tienda.fijar_stock(a, 0)`.


In [ ]:
#@title Preparar la conversación
if EN_COLAB:
    import asistente
    import base64
    import re
    from dataclasses import asdict
    from html import escape
    from google.colab import output
    from asistente import Asistente
    from tienda import Tienda

    # El ASR puede omitir el artículo o pluralizar «compra» en una confirmación breve.
    asistente.CONFIRMA = re.compile(
        r"(?:si[,]?\s*)?(?:confirmo(?:\s+(?:(?:la )?compra|(?:las )?compras|el pedido|la cotizacion))?"
        r"|acepto(?:\s+la cotizacion)?)"
    )

    def mostrar_caso(nombre):
        path = RESULTADOS / "operacion" / nombre / "voz" / "traza.json"
        traza = json.loads(path.read_text(encoding="utf-8"))
        print(nombre, "| Cumplimiento:", traza["correcto"])
        for paso in traza["turnos"]:
            print("Usuario:", paso["texto"])
            display(Audio(filename=paso["audio"]["audio_entrada"]))
            print("Asistente:", paso["respuesta"])
            display(Audio(filename=paso["audio"]["audio_respuesta"]))
            print("Segundos desde el archivo:", round(paso["tiempos_s"]["archivo_a_audio_listo"], 2))
        print("Pedidos:", traza["estado_final"]["pedidos"])

    def reiniciar_demo():
        global demo_tienda, demo_agente
        if "demo_tienda" in globals():
            demo_tienda.cerrar()
        demo_tienda = Tienda(indice.fichas, SEED)
        demo_tienda.restaurar({a: (3000, 5), b: (4400, 6), c: (5900, 4)}, stock_otros=0)
        demo_agente = Asistente(indice, demo_tienda, planificador, sesion="demostracion")
        print("Sesión iniciada. Producto de la demostración:", familia)

    def _turno_audio(ruta, *, mostrar_texto=True):
        desde = perf_counter()
        asr = whisper.transcribir(ruta)
        asr_s = perf_counter() - desde
        texto = asr["texto"]
        if texto.strip():
            paso = demo_agente.procesar(texto)
        else:
            paso = {
                "texto": "", "estado": "sin_voz", "origen_plan": "asr",
                "plan": asdict(asistente.Plan()), "memoria": asdict(demo_agente.memoria),
                "respuesta": "No pude transcribir ese turno. Vuelve a grabarlo.",
                "evidencia": {}, "tiempos_s": {},
            }
        inicio_tts = perf_counter()
        wav = RESULTADOS / "demo" / f"respuesta_{uuid4().hex[:10]}.wav"
        hablado = voz.sintetizar(paso["respuesta"], wav)
        paso["tiempos_s"].update({"asr": asr_s, "tts_y_wav": perf_counter() - inicio_tts,
                                  "archivo_a_audio_listo": perf_counter() - desde})
        paso.update({"archivo_entrada": str(ruta), "archivo_respuesta": str(wav), "texto_sintetizado": hablado})
        if mostrar_texto:
            print("Transcripción:", texto)
            print("Asistente:", paso["respuesta"])
        return paso, wav

    def _respuesta_html(paso):
        respuesta = paso["respuesta"]
        opciones = paso["evidencia"].get("opciones", [])
        if not opciones:
            return f'<div style="white-space:pre-wrap">{escape(respuesta)}</div>'
        prefijo = respuesta.removesuffix(Asistente._decir_opciones(opciones)).strip()
        parrafo = f"<div>{escape(prefijo)}</div>" if prefijo else ""
        filas = "".join(
            f'<li value="{int(o["opcion"])}" style="margin:.45rem 0">'
            f'<strong>{escape(o["titulo"])}</strong><br>'
            f'{escape(asistente.soles(o["precio"]))} por unidad · Stock {int(o["stock"])}</li>'
            for o in opciones
        )
        return f'{parrafo}<ol style="padding-left:1.6rem;margin:.35rem 0 .65rem">{filas}</ol>'

    def _mostrar_turno(paso, wav, *, numero=None, fin=None):
        titulo = f'<h4 style="margin:0 0 .6rem">Turno {numero}</h4>' if numero is not None else ""
        instruccion = escape(paso["texto"].strip() or "Sin transcripción")
        contenido = (
            f'{titulo}<p style="margin:0 0 .65rem"><strong>Instrucción:</strong> {instruccion}</p>'
            f'<strong>Respuesta:</strong>{_respuesta_html(paso)}'
            '<audio controls preload="auto" aria-label="Audio de la respuesta" '
            'style="display:block;width:20rem;max-width:100%;margin:.6rem 0 .25rem"></audio>'
            '<small class="tiempo" style="display:block"></small>'
        )
        datos = {
            "contenido": contenido,
            "archivo": "data:audio/wav;base64," + base64.b64encode(Path(wav).read_bytes()).decode("ascii"),
            "fin": fin,
        }
        return output.eval_js(r"""
        (({contenido, archivo, fin}) => new Promise(resolver => {
          document.getElementById('lab4-controles-voz')?.remove();
          const turno = document.createElement('section');
          turno.style.cssText = 'box-sizing:border-box;max-width:54rem;margin:.7rem 0;padding:.85rem 1rem;border:1px solid #8887;border-radius:.6rem;line-height:1.5;overflow-wrap:anywhere';
          turno.innerHTML = contenido;
          const audio = turno.querySelector('audio');
          const tiempo = turno.querySelector('.tiempo');
          const espera = setTimeout(() => resolver(null), 10000);
          audio.addEventListener('canplay', () => {
            clearTimeout(espera);
            const listo = Date.now();
            if (fin !== null) {
              tiempo.textContent = 'Tiempo hasta el audio: ' + ((listo - fin) / 1000).toFixed(2) + ' s';
            }
            resolver(listo);
          }, {once: true});
          audio.addEventListener('error', () => {
            clearTimeout(espera);
            tiempo.textContent = 'No se pudo reproducir el audio de este turno.';
            resolver(null);
          }, {once: true});
          document.body.appendChild(turno);
          audio.src = archivo;
        }))(
        """ + json.dumps(datos) + ")")

    def turno_wav(ruta):
        paso, wav = _turno_audio(Path(ruta), mostrar_texto=False)
        _mostrar_turno(paso, wav)
        return paso

    JS_GRABAR = r"""
    new Promise(resolver => {
      const panel = document.createElement('div');
      panel.id = 'lab4-controles-voz';
      panel.style.cssText = 'margin:.8rem 0; padding:.25rem 0';
      const estado = document.createElement('p');
      estado.setAttribute('role', 'status');
      estado.style.margin = '0 0 .5rem';
      estado.textContent = 'Pulsa Grabar turno y habla cuando aparezca el aviso de grabación.';
      const grabar = document.createElement('button');
      grabar.textContent = 'Grabar turno';
      grabar.style.marginRight = '.6rem';
      const salir = document.createElement('button');
      salir.textContent = 'Finalizar conversación';
      salir.onclick = () => { panel.remove(); resolver({salir: true}); };
      panel.append(estado, grabar, salir);
      document.body.appendChild(panel);
      panel.scrollIntoView({block: 'nearest'});

      async function iniciar() {
        grabar.disabled = salir.disabled = true;
        estado.textContent = 'Abriendo el micrófono…';
        let stream;
        function detenerPistas() { stream?.getTracks().forEach(pista => pista.stop()); }
        function reintentar(mensaje) {
          detenerPistas();
          estado.textContent = mensaje;
          grabar.textContent = 'Grabar turno';
          grabar.onclick = iniciar;
          grabar.disabled = salir.disabled = false;
        }
        try {
          stream = await navigator.mediaDevices.getUserMedia({audio: true});
          const recorder = new MediaRecorder(stream);
          const partes = [];
          let fin;
          recorder.ondataavailable = evento => partes.push(evento.data);
          recorder.onerror = () => reintentar('La grabación se interrumpió. Vuelve a intentarlo.');
          recorder.onstop = () => {
            detenerPistas();
            if (fin === undefined) return;
            const audio = new Blob(partes, {type: recorder.mimeType});
            if (!audio.size) { reintentar('La grabación quedó vacía. Inténtalo otra vez.'); return; }
            const reader = new FileReader();
            reader.onerror = () => reintentar('No se pudo leer la grabación. Inténtalo otra vez.');
            reader.onload = () => {
              grabar.remove();
              salir.remove();
              estado.textContent = 'Procesando tu turno…';
              resolver({data: reader.result, fin});
            };
            reader.readAsDataURL(audio);
          };
          recorder.start();
          estado.textContent = 'Grabando. Habla ahora.';
          grabar.textContent = 'Terminar grabación';
          grabar.disabled = false;
          grabar.onclick = () => {
            fin = Date.now();
            grabar.disabled = true;
            recorder.stop();
          };
        } catch (error) {
          reintentar('No se pudo abrir el micrófono. Revisa el permiso e inténtalo otra vez.');
        }
      }
      grabar.onclick = iniciar;
    });
    """

    def turno_microfono(numero=None):
        datos = output.eval_js(JS_GRABAR)
        if datos.get("salir"):
            return None
        uri = datos["data"]
        mime, contenido = uri.split(",", 1)
        extension = ".ogg" if "ogg" in mime else ".webm"
        entrada = RESULTADOS / "demo" / f"entrada_{uuid4().hex[:10]}{extension}"
        entrada.parent.mkdir(parents=True, exist_ok=True)
        entrada.write_bytes(base64.b64decode(contenido))
        paso, wav = _turno_audio(entrada, mostrar_texto=False)
        listo = _mostrar_turno(paso, wav, numero=numero, fin=datos["fin"])
        if listo is not None:
            latencia = (listo - datos["fin"]) / 1000
            paso["tiempos_s"]["fin_grabacion_a_reproduccion"] = latencia
            paso["tiempos_s"]["captura_y_entrega"] = max(
                0, latencia - paso["tiempos_s"]["archivo_a_audio_listo"],
            )
        if numero is not None:
            paso["turno"] = numero
        registro = RESULTADOS / "demo" / "turnos.jsonl"
        with registro.open("a", encoding="utf-8") as archivo:
            archivo.write(json.dumps(paso, ensure_ascii=False) + "\n")
        return paso

    def empaquetar_evidencias():
        return shutil.make_archive(str(ROOT / "Evidencias_Laboratorio4"), "zip", ROOT, str(RESULTADOS))

    def descargar_evidencias():
        from google.colab import files
        files.download(empaquetar_evidencias())

    print("Funciones listas para configurar la voz y conversar por micrófono.")


In [ ]:
#@title Configurar mi voz para las respuestas
#@markdown Graba una toma, comprueba que el texto coincida con lo que dijiste y escucha una frase nueva con tu voz.
from copy import copy
from io import BytesIO
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4
import base64
import json
import numpy as np
import soundfile as sf
from faster_whisper.audio import decode_audio
from google.colab import output
from IPython.display import display, Markdown
from esci import hash_archivo
from evaluacion import word_error_rate

TEXTO_LECTURA = (
    "Esta mañana salí temprano a caminar por el barrio. Al volver, preparé café, abrí la ventana "
    "y escuché cómo empezaba a llover. Más tarde, guardé el paraguas azul y llamé a mi hermana "
    "para contarle lo que había pasado."
)
TEXTO_PRUEBA_VOZ = "Encontré dos mantas eléctricas dentro de tu presupuesto. ¿Quieres que añada una al carrito?"

JS_REVISAR_REFERENCIA = r"""
(({audio, texto}) => new Promise(resolver => {
  document.getElementById('lab4-controles-voz')?.remove();
  const panel = document.createElement('section');
  panel.style.cssText = 'max-width:52rem;margin:.75rem 0;line-height:1.5';
  panel.innerHTML = '<p><strong>Revisa tu grabación</strong></p>' +
    '<audio controls aria-label="Escuchar la toma" style="display:block;margin:.5rem 0"></audio>' +
    '<label>Comprueba que el texto coincida con lo que dijiste.' +
    '<textarea rows="3" style="display:block;width:98%;margin:.35rem 0 .7rem"></textarea></label>' +
    '<button>Clonar esta voz</button> <button>Repetir grabación</button>';
  panel.querySelector('audio').src = audio;
  const campo = panel.querySelector('textarea');
  campo.value = texto;
  const [clonar, repetir] = panel.querySelectorAll('button');
  clonar.onclick = () => {
    if (!campo.value.trim()) { campo.focus(); return; }
    panel.remove();
    resolver({texto: campo.value.trim()});
  };
  repetir.onclick = () => { panel.remove(); resolver({repetir: true}); };
  document.body.appendChild(panel);
  panel.scrollIntoView({block: 'nearest'});
}))(
"""


def configurar_voz_por_microfono():
    global referencia, metadatos_voz
    if not all(nombre in globals() for nombre in ("voz", "whisper", "JS_GRABAR", "_mostrar_turno")):
        raise RuntimeError("Ejecuta la carga de voz de la sección 4 y Preparar la conversación antes de grabar.")
    grabador = (JS_GRABAR
        .replace("new Promise(resolver => {", "new Promise(resolver => { document.getElementById('lab4-controles-voz')?.remove();", 1)
        .replace("Grabar turno", "Grabar mi voz")
        .replace("Finalizar conversación", "Usar la voz actual")
        .replace("Procesando tu turno…", "Transcribiendo tu grabación…"))
    display(Markdown("**Lee este texto a ritmo natural**\n\n> " + TEXTO_LECTURA))
    while True:
        datos = output.eval_js(grabador)
        if datos.get("salir"):
            return
        uri = datos["data"]
        muestras = decode_audio(BytesIO(base64.b64decode(uri.split(",", 1)[1])), sampling_rate=24000)
        if len(muestras) < 72000 or not np.any(np.abs(muestras) > .001):
            print("La toma quedó demasiado corta o en silencio. Vuelve a grabar el párrafo.")
            continue
        with TemporaryDirectory() as temporal:
            toma = Path(temporal) / "toma.wav"
            sf.write(toma, muestras, 24000, subtype="PCM_16")
            detectado = whisper.transcribir(toma)["texto"].strip()
        if not detectado:
            print("No se detectó voz con claridad. Vuelve a grabar en un lugar silencioso.")
            continue
        sugerido = TEXTO_LECTURA if word_error_rate(TEXTO_LECTURA, detectado) <= .15 else detectado
        revision = output.eval_js(JS_REVISAR_REFERENCIA + json.dumps({"audio": uri, "texto": sugerido}) + ")")
        if revision.get("repetir"):
            continue
        texto = revision["texto"]
        break

    print("Generando una frase nueva con tu voz…", flush=True)
    carpeta = ROOT / "referencias_autorizadas" / uuid4().hex[:12]
    carpeta.mkdir(parents=True)
    ruta = carpeta / "referencia_asistente.wav"
    muestra = carpeta / "respuesta_de_prueba.wav"
    sf.write(ruta, muestras, 24000, subtype="PCM_16")
    datos_ref = {
        "origen": "Grabación propia autorizada, capturada con el micrófono en Colab",
        "texto": texto, "frecuencia_hz": 24000,
        "duracion_s": len(muestras) / 24000, "sha256": hash_archivo(ruta),
    }
    (carpeta / "referencia_asistente.json").write_text(json.dumps(datos_ref, ensure_ascii=False, indent=2), encoding="utf-8")

    # Se comparte el modelo cargado y se valida la nueva voz antes de activarla.
    candidata = copy(voz)
    candidata.referencia = ruta
    candidata.texto_referencia = texto
    candidata.prompt = voz.modelo.create_voice_clone_prompt(ref_audio=str(ruta), ref_text=texto, x_vector_only_mode=False)
    hablado = type(voz).sintetizar(candidata, TEXTO_PRUEBA_VOZ, muestra)
    similitud = type(voz).similitud(candidata, muestra)
    reconocido = whisper.transcribir(muestra)["texto"]
    error = word_error_rate(hablado, reconocido)
    (carpeta / "prueba_voz.json").write_text(json.dumps({
        "texto_esperado": hablado, "texto_asr": reconocido,
        "wer_asr": error, "similitud_coseno": similitud,
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    voz.referencia, voz.texto_referencia, voz.prompt = ruta, texto, candidata.prompt
    referencia, metadatos_voz = ruta, datos_ref
    _mostrar_turno({"texto": "Prueba de la voz de las respuestas", "respuesta": hablado, "evidencia": {}}, muestra)
    print(f"Voz propia activa para la conversación. Referencia de {datos_ref['duracion_s']:.1f} s.")
    print(f"Prueba de una frase: similitud de voz {similitud:.3f} y WER por ASR {error:.1%}.")


configurar_voz_por_microfono()


In [ ]:
#@title Conversación por micrófono
#@markdown Graba un turno y revisa la instrucción transcrita y su respuesta en texto y audio.
if "_mostrar_turno" not in globals() or "voz" not in globals():
    raise RuntimeError("Ejecuta la carga de voz de la sección 4 y Preparar la conversación.")

reiniciar_demo()
numero = 1
while turno_microfono(numero) is not None:
    numero += 1
print("Conversación finalizada.")


## 6 · Evaluación operativa y de voz

Ejecuta esta celda después de pulsar **Finalizar conversación**. La evaluación usa la voz de respuesta activa en la sesión. Para evaluar sin usar el micrófono, ejecuta hasta **Preparar la conversación** y continúa aquí con la referencia cargada en la sección 4.

Cada escenario restaura su propia tienda. Los seis casos con audio también se ejecutan con el texto correcto. La traza guarda la transcripción, las operaciones, los audios y los estados inicial y final.

Los guiones registran las cantidades, los importes y los productos esperados. El caso de corrección registra la cantidad reconocida y las cotizaciones antes y después del cambio. El resumen también calcula el error de entidades al excluir el turno con enmascaramiento. El caso de stock modifica las existencias entre la cotización y la confirmación y verifica que repetir la confirmación conserve el mismo pedido.

La tasa de error de palabras estima la inteligibilidad de las respuestas. El coseno entre representaciones de locutor estima la semejanza con la referencia. La latencia automática se mide desde que está listo el archivo de entrada. La demostración por micrófono mide además el tramo desde el fin de la grabación.

Para escuchar un caso evaluado, usa `mostrar_caso("01_compra_voz")` o `mostrar_caso("06_stock_y_repeticion_voz")` en una celda de Colab. Usa `descargar_evidencias()` para descargar el ZIP con los resultados de la sesión.


In [ ]:
#@title Ejecutar la evaluación operativa y de voz
if EN_COLAB:
    if "voz" not in globals() or "empaquetar_evidencias" not in globals():
        raise RuntimeError("Ejecuta la carga de voz de la sección 4 y Preparar la conversación de la sección 5.")
    from evaluacion import ejecutar_escenarios

    torch.manual_seed(SEED)

    def casos_con_avance():
        for numero, caso in enumerate(casos, 1):
            print(f"[{numero:02d}/{len(casos)}] {caso.nombre}", flush=True)
            yield caso

    escenarios, turnos = ejecutar_escenarios(
        casos_con_avance(), indice, anclas, planificador, RESULTADOS / "operacion",
        whisper, voz, ENTRADAS, FUENTE_ENTRADA,
    )
    display(escenarios)
    print(f"Escenarios correctos: {escenarios.correcto.sum()}/{len(escenarios)}")
    print("Incumplimientos:", int(escenarios.incumplimientos.sum()))
    if not turnos.empty:
        entidad = ["cantidad", "importe", "producto"]
        display(pd.DataFrame({
            "ocurrencias": [int(turnos[f"entidad_{x}"].sum()) for x in entidad],
            "errores": [int(turnos[f"error_{x}"].sum()) for x in entidad],
        }, index=entidad))
        campos_voz = ["asr", "planificacion", "recuperacion", "recomendacion_llm", "tienda_y_respuesta", "tts_y_wav",
                      "archivo_a_audio_listo", "wer_entrada", "wer_respuesta", "similitud_vocal"]
        resumen_turnos = turnos[campos_voz].describe(percentiles=[0.5, 0.95]).loc[["count", "mean", "50%", "95%"]]
        display(resumen_turnos.round(3))
        resumen_turnos.to_csv(RESULTADOS / "resumen_medidas_de_voz.csv")

    versiones = {}
    for paquete in ("torch", "torchaudio", "numpy", "pandas", "pyarrow", "faster-whisper",
                    "transformers", "qwen-tts", "soundfile", "matplotlib", "av"):
        try:
            versiones[paquete] = metadata.version(paquete)
        except metadata.PackageNotFoundError:
            versiones[paquete] = "ausente"
    gpu = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
    configuracion = {
        "id_ejecucion": ID_EJECUCION, "plataforma": "Google Colab", "sha256_fuentes": SHA_FUENTES,
        "python": platform.python_version(), "versiones": versiones, "cpu_logicas": os.cpu_count(),
        "sha256_modelos": sha256((ROOT / "modelos.py").read_bytes()).hexdigest(),
        "dtype_clonacion": str(voz.modelo.model.dtype),
        "entradas_sinteticas_reutilizadas_de": ORIGEN_ENTRADAS,
        "datos_sha256": datos.resumen["sha256_preparados"],
        "gpu": None if gpu is None else gpu.name,
        "vram_gib": None if gpu is None else gpu.total_memory / 1024**3,
        "vram_pico_gib": None if gpu is None else torch.cuda.max_memory_allocated() / 1024**3,
        "cuda": torch.version.cuda, "device": equipo, "dtype": str(dtype_modelo(equipo)), "semilla": SEED,
        "almacenamiento_libre_gib": shutil.disk_usage(ROOT).free / 1024**3,
        "clonacion_decodificacion": {"do_sample": True, "non_streaming_mode": True,
                                     "max_new_tokens": "max(96, 36 + 7 * palabras)"},
        "modelos": {"recuperacion": E5_ID, "agente": AGENTE_ID, "asr": ASR_ID,
                    "diseno": DISENO_ID, "clonacion": CLON_ID},
        "revision_transformers": {"e5": getattr(encoder.model.config, "_commit_hash", None),
                                   "agente": getattr(planificador.model.config, "_commit_hash", None)},
        "carga_inicial_s": {"e5": encoder.carga_s, "indice": indice_s, "asr": whisper.carga_s,
                            "agente": planificador.carga_s, "clonacion": voz.carga_s,
                            "diseno": metadatos_voz.get("carga_modelo_s")},
        "recuperacion": {"bm25_k1": 1.5, "bm25_b": 0.75, "rrf_k": 60,
                         "coseno_minimo_interaccion": 0.82, "e5": encoder.config},
        "voz_referencia": metadatos_voz, "entradas_voz": FUENTE_ENTRADA,
        "interpretacion_llm": {"llamadas": planificador.llamadas, "recuperaciones_por_reglas": planificador.recuperaciones,
                               "origen_plan_en_audio": turnos["origen_plan"].value_counts().to_dict() if not turnos.empty else {}},
    }
    (RESULTADOS / "configuracion.json").write_text(
        json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    display(pd.Series(configuracion["carga_inicial_s"], name="segundos").to_frame().round(3))
    print("Evidencias guardadas en:", empaquetar_evidencias())


## Referencias y límites

- Reddy et al. (2022), *Shopping Queries Dataset: A Large-Scale ESCI Benchmark for Improving Product Search*. [Artículo](https://arxiv.org/abs/2206.06588) · [Código y datos](https://github.com/amazon-science/esci-data)
- [E5 multilingüe](https://huggingface.co/intfloat/multilingual-e5-small) · [Qwen2.5](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) · [Qwen3‑TTS](https://github.com/QwenLM/Qwen3-TTS) · [Faster‑Whisper](https://github.com/SYSTRAN/faster-whisper)

El catálogo describe un mercado específico y presenta faltantes en color y descripción. Las entradas sintéticas controlan el contenido de los ensayos, y caracterizar habla espontánea requeriría grabaciones representativas. El video usa las salidas y los tiempos observados al ejecutar los modelos en Colab.


In [ ]:
#@title Prueba de captura sintética en el navegador
from google.colab import output


def probar_captura_sintetica():
    origen = RESULTADOS / "entradas" / "02_aclaracion_voz" / "entrada_01.wav"
    carpeta = ROOT / "pruebas_captura" / uuid4().hex[:8]
    carpeta.mkdir(parents=True)
    archivo_wav = base64.b64encode(origen.read_bytes()).decode("ascii")
    reiniciar_demo()
    captura = output.eval_js(r"""
    ((wav) => new Promise((ok, error) => {
      const ahora = () => performance.timeOrigin + performance.now();
      const boton = document.createElement('button');
      boton.textContent = 'Iniciar captura sintética';
      document.body.appendChild(boton);
      boton.onclick = async () => {
        boton.disabled = true;
        boton.textContent = 'Grabando la entrada sintética';
        const contexto = new AudioContext();
        try {
          await contexto.resume();
          const bytes = Uint8Array.from(atob(wav), c => c.charCodeAt(0));
          const audio = await contexto.decodeAudioData(bytes.buffer);
          const fuente = contexto.createBufferSource();
          const destino = contexto.createMediaStreamDestination();
          fuente.buffer = audio;
          fuente.connect(destino);
          const grabador = new MediaRecorder(destino.stream);
          const trozos = [];
          let fin = 0;
          grabador.ondataavailable = e => trozos.push(e.data);
          grabador.onerror = () => { contexto.close(); error('Error al grabar la fuente sintética.'); };
          grabador.onstop = () => {
            const lector = new FileReader();
            lector.onerror = () => { contexto.close(); error('No se pudo leer la captura.'); };
            lector.onloadend = () => {
              boton.remove();
              contexto.close();
              ok({data: lector.result, fin, duracion_fuente_s: audio.duration});
            };
            lector.readAsDataURL(new Blob(trozos, {type: grabador.mimeType}));
          };
          fuente.onended = () => { fin = ahora(); grabador.stop(); };
          grabador.start();
          fuente.start(contexto.currentTime + 0.15);
        } catch (e) {
          contexto.close();
          boton.remove();
          error(String(e));
        }
      };
    }))(
    """ + json.dumps(archivo_wav) + ")")
    mime, datos_grabados = captura["data"].split(",", 1)
    entrada = carpeta / ("captura.ogg" if "ogg" in mime else "captura.webm")
    entrada.write_bytes(base64.b64decode(datos_grabados, validate=True))
    shutil.copy2(origen, carpeta / "fuente_sintetica.wav")
    print("Captura sintética completada. Procesando la respuesta.", flush=True)
    paso, salida_wav = _turno_audio(entrada)
    audio_respuesta = "data:audio/wav;base64," + base64.b64encode(salida_wav.read_bytes()).decode("ascii")
    listo = output.eval_js(r"""
    ((fuente) => new Promise((ok, error) => {
      const audio = document.createElement('audio');
      audio.controls = true;
      audio.preload = 'auto';
      audio.addEventListener('canplay', () => ok(performance.timeOrigin + performance.now()), {once: true});
      audio.addEventListener('error', () => error('El navegador no pudo cargar la respuesta.'), {once: true});
      document.body.appendChild(audio);
      audio.src = fuente;
    }))(
    """ + json.dumps(audio_respuesta) + ")")
    total_s = (listo - captura["fin"]) / 1000
    assert total_s >= paso["tiempos_s"]["archivo_a_audio_listo"]
    paso["tiempos_s"]["fin_grabacion_a_reproduccion"] = total_s
    paso["tiempos_s"]["captura_y_entrega"] = total_s - paso["tiempos_s"]["archivo_a_audio_listo"]
    reporte = {
        "origen_entrada": "fuente sintética grabada con MediaRecorder en el navegador",
        "origen_referencia_tts": configuracion["voz_referencia"]["origen"],
        "duracion_fuente_s": captura["duracion_fuente_s"],
        "transcripcion": paso["texto"], "respuesta": paso["respuesta"],
        "estado": paso["estado"], "tiempos_s": paso["tiempos_s"],
        "id_ejecucion_base": RESULTADOS.name,
    }
    assert reporte["estado"] == "aclaracion", reporte["estado"]
    (carpeta / "medicion.json").write_text(json.dumps(reporte, ensure_ascii=False, indent=2), encoding="utf-8")
    shutil.copy2(salida_wav, carpeta / "respuesta_clonada.wav")
    print(f"Fin de captura sintética a respuesta reproducible: {total_s:.2f} s")
    display(pd.Series(reporte["tiempos_s"], name="segundos").to_frame().round(3))
    display(Audio(filename=str(salida_wav)))
    print("Evidencia:", shutil.make_archive(str(ROOT / "Captura_Sintetica_Lab4"), "zip", carpeta))
    return reporte


captura_sintetica_lab4 = probar_captura_sintetica()
